# Face Hair - automatyczny transfer wag z Debiana
Notebook instaluje DiffLocks (jesli potrzeba), pobiera wagi bezposrednio z Twojego Debiana i uruchamia generator. 
W drugiej komorce wklej tylko prywatny link otrzymany w rozmowie. Nie wysylaj go publicznie.
Srodowisko: GPU T4. Transfer jest wznawialny i nie wymaga uploadu przez przegladarke.


In [ ]:
from pathlib import Path
import os, sys
if Path("/content/facehair_env/bin/python").exists() and Path("/content/difflocks_launcher.py").exists():
    print("Srodowisko juz zainstalowane. Pomijam ponowna instalacje.")
else:
    import os, sys, subprocess, urllib.request
    from pathlib import Path

    def run(*args, env=None):
        subprocess.run(args, check=True, env=env)

    print("Face Hair - naprawiony start DiffLocks Studio")
    run("nvidia-smi")
    run(sys.executable, "-m", "pip", "install", "-q", "uv")
    run("uv", "python", "install", "3.12")
    run("uv", "venv", "--python", "3.12", "--seed", "/content/facehair_env")

    env = os.environ.copy()
    env["PATH"] = "/content/facehair_env/bin:" + env.get("PATH", "")
    env["PYTHONNOUSERSITE"] = "1"
    env.pop("PYTHONPATH", None)
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            env["HF_TOKEN"] = token
    except Exception:
        pass

    python = "/content/facehair_env/bin/python"
    run(python, "-m", "pip", "install", "-q", "ipywidgets", "ipython", "requests", "huggingface_hub", env=env)

    url = "https://raw.githubusercontent.com/arqdariogomez/DiffLocks-Studio/3db8c8f/launchers/Difflocks_colab_github_minimal.txt"
    source = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
    source = source.replace("https://shi-labs.com/natten/wheels/", "https://whl.natten.org/old")
    start = source.index("def render_debug_html(logs_list):")
    end = source.index("\n" + chr(39) * 3 + "\n", start)
    source = source[:start] + source[start:end].replace(chr(39) * 3, chr(34) * 3) + source[end:]
    import ast
    ast.parse(source)
    print("Poprawka skladni DiffLocks: OK", flush=True)
    source = source.replace("pip install --no-cache-dir torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0", "pip install --no-cache-dir torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0")
    source = source.replace('HF_ASSETS_REPO = "arqdariogomez/difflocks-assets-hybrid"', 'HF_ASSETS_REPO = "arqdariogomez/difflocks-assets-hybrid"')
    file = Path("/content/Face_DiffLocks_Launcher.py")
    source = source.replace('        render_status(96, "🚀 Starting DiffLocks Studio...")', '        render_status(100, "Srodowisko gotowe. Przejdz do kolejnej komorki.", "success")\n        return')
    file.write_text(source)
    print("Przygotowuje srodowisko. To moze chwile potrwac.")
    run(python, "-u", str(file), env=env)

    

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import requests, time, sys

root = Path("/content/DiffLocks-Studio")
root.mkdir(parents=True, exist_ok=True)
zip_path = Path("/content/difflocks_checkpoints.zip")
ckpt = root / "checkpoints"
models = list(ckpt.rglob("*.pth"))
codec = list(ckpt.rglob("strand_codec.pt"))

if models and codec:
    print("Wagi DiffLocks juz sa na miejscu. Pomijam transfer.")
else:
    url = input("Wklej prywatny link z rozmowy i nacisnij Enter: ").strip()
    if not url.startswith("https://") or "/facehair?key=" not in url:
        raise ValueError("Wklej pelny adres HTTPS otrzymany w rozmowie.")
    session = requests.Session()
    response = session.head(url, timeout=45)
    response.raise_for_status()
    expected = int(response.headers["Content-Length"])
    print(f"Do pobrania {expected / 1024**3:.2f} GiB bezposrednio z Twojego Debiana.", flush=True)
    if zip_path.exists() and zip_path.stat().st_size > expected:
        raise RuntimeError("Lokalny plik jest wiekszy od zrodla. Nie nadpisuje go.")
    offset = zip_path.stat().st_size if zip_path.exists() else 0
    if offset == expected:
        print("Archiwum juz pobrane. Sprawdzam.")
    else:
        chunk_size = 64 * 1024 * 1024
        last_percent = -1
        while offset < expected:
            end = min(offset + chunk_size, expected) - 1
            for attempt in range(6):
                try:
                    with session.get(url, headers={"Range": f"bytes={offset}-{end}"}, stream=True, timeout=(35, 140)) as res:
                        if res.status_code != 206:
                            raise RuntimeError(f"Serwer nie obsluguje wznawiania: HTTP {res.status_code}")
                        if res.headers.get("Content-Range", "").split(" ",1)[-1].split("-")[0] != str(offset):
                            raise RuntimeError("Nieoczekiwany zakres danych.")
                        with zip_path.open("ab") as target:
                            before=target.tell()
                            for data in res.iter_content(2 * 1024 * 1024):
                                if data: target.write(data)
                            received=target.tell()-before
                    if received != end-offset+1:
                        raise IOError(f"Niepelny fragment: {received} z {end-offset+1} bajtow")
                    offset = zip_path.stat().st_size
                    percent = int(offset * 100 / expected)
                    if percent // 5 != last_percent // 5 or percent == 100:
                        print(f"Transfer: {percent}% ({offset / 1024**3:.2f} / {expected / 1024**3:.2f} GiB)", flush=True)
                        last_percent = percent
                    break
                except (requests.RequestException, OSError, IOError) as e:
                    with zip_path.open("r+b") as target: target.truncate(offset)
                    if attempt == 5: raise
                    print(f"Ponawiam fragment po bledzie: {str(e)[:100]}", flush=True)
                    time.sleep(2 + attempt)
    print("Transfer zakonczony. Sprawdzam archiwum.", flush=True)
    with ZipFile(zip_path) as z:
        names = [info.filename for info in z.infolist()]
        if not any(n.endswith("strand_codec.pt") for n in names) or not any(n.endswith(".pth") for n in names):
            raise RuntimeError("Archiwum nie zawiera wymaganych wag.")
        for info in z.infolist():
            parts=Path(info.filename).parts
            if Path(info.filename).is_absolute() or ".." in parts:
                raise RuntimeError("Nieprawidlowa sciezka w archiwum.")
        z.extractall(root)
    models = list(ckpt.rglob("*.pth"))
    codec = list(ckpt.rglob("strand_codec.pt"))
    if not models or not codec:
        raise RuntimeError("Po rozpakowaniu brakuje wag.")
    print("GOTOWE: wagi AI rozpakowane. Przechodze do generatora.", flush=True)


In [ ]:
from pathlib import Path
import os, subprocess, re
from IPython.display import display, HTML
python=Path("/content/facehair_env/bin/python")
launcher=Path("/content/difflocks_launcher.py")
if not python.exists() or not launcher.exists():
    raise RuntimeError("Brakuje srodowiska. Uruchom pierwsza komorke.")
if not list(Path("/content/DiffLocks-Studio/checkpoints").rglob("strand_codec.pt")):
    raise RuntimeError("Brakuje wag AI. Uruchom druga komorke.")
env=os.environ.copy()
env["PATH"]="/content/facehair_env/bin:"+env.get("PATH","")
env["PYTHONNOUSERSITE"]="1"
env.pop("PYTHONPATH",None)
print("Uruchamiam DiffLocks Studio. Otworz link gradio.live, gdy sie pojawi.")
proc=subprocess.Popen([str(python), "-u", str(launcher)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line,end="",flush=True)
    match=re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live",line)
    if match:
        display(HTML('<p><a href="'+match.group(0)+'" target="_blank">OTWORZ GENERATOR WLOSOW</a></p>'))
code=proc.wait()
if code:
    raise RuntimeError(f"DiffLocks zakonczyl sie bledem {code}, szczegoly sa powyzej.")
